# Speech Interfaces & TTS — Practical Notebook

This practical turns a text agent into a spoken agent.

You will build a reusable pipeline:

```text
audio input → speech-to-text → speech-safe LLM response → text-to-speech → output audio
```

This builds on earlier work with LLM agents, streaming systems, and production reliability. The focus here is the speech layer: provider choices, audio quality, latency, and output that can later drive an avatar.


## Setup

Run the notebook once with mock providers first. Mock providers do not call paid APIs, which makes the classroom demo reliable even before keys or microphones are configured.

After the offline run works, switch individual providers to OpenAI, ElevenLabs, Piper, or local Whisper.


In [12]:
from pathlib import Path
from IPython.display import Audio, display

from audio_utils import create_mock_speech_wav, measure_call
from stt_providers import MockSTTProvider, OpenAITranscriptionProvider, LocalWhisperProvider
from tts_providers import MockTTSProvider, OpenAITTSProvider, ElevenLabsTTSProvider, PiperTTSProvider
from voice_agent import VoiceAgent
from evaluation_rubric import create_rubric_csv, explain_scoring

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

sample_audio = DATA_DIR / "sample_voice_question.wav"
if not sample_audio.exists():
    create_mock_speech_wav(sample_audio)

print(f"Sample audio: {sample_audio}")
print(f"Rubric: {create_rubric_csv(DATA_DIR / 'tts_evaluation_rubric.csv')}")
print(explain_scoring())


Sample audio: data\sample_voice_question.wav
Rubric: data\tts_evaluation_rubric.csv
Score each dimension from 1 to 5. 1 means unacceptable for a user-facing assistant; 3 means usable for a prototype; 5 means strong enough for a polished product demo.


## Listen to the sample input

The included sample asks a short question about TTS. Use it as the stable fallback. In a live class, you can later replace it with microphone audio.


In [2]:
display(Audio(filename=str(sample_audio)))


## 1. Start with the provider contract

The key design decision is simple: every STT provider returns the same kind of transcript object, and every TTS provider returns the same kind of audio object.

That lets us swap providers without rewriting the agent.


In [3]:
mock_stt = MockSTTProvider()
mock_transcript = mock_stt.transcribe(sample_audio)

print(mock_transcript)
print("Transcript text:", mock_transcript.text)


STTResult(text='Can you explain in simple words how text to speech works for an AI assistant?', provider='mock-stt', model='fixed-transcript', latency_ms=None)
Transcript text: Can you explain in simple words how text to speech works for an AI assistant?


## 2. Try cloud STT with OpenAI

This cell uses OpenAI only when `OPENAI_API_KEY` is configured in `.env`.

If the key is missing, the cell explains the issue and continues. That is intentional for classroom robustness.


In [4]:
try:
    openai_stt = OpenAITranscriptionProvider()
    timed_stt = measure_call(openai_stt.transcribe, sample_audio)
    openai_transcript = timed_stt.value
    openai_transcript.latency_ms = timed_stt.elapsed_ms

    print(openai_transcript)
    print(f"Latency: {openai_transcript.latency_ms:.0f} ms")
except Exception as exc:
    print("OpenAI STT skipped:", exc)
    openai_transcript = mock_transcript


STTResult(text='Can you explain in simple words how text-to-speech works for an AI assistant?', provider='openai-stt', model='gpt-4o-mini-transcribe', latency_ms=8912.004000005254)
Latency: 8912 ms


## 3. Optional: local Whisper

Local Whisper is useful when privacy and offline operation matter. It is optional because it can require larger downloads, ffmpeg, and CPU/GPU time.

Install only when the machine can handle it:

```bash
pip install -r requirements_local_whisper_optional.txt
```


In [ ]:
RUN_LOCAL_WHISPER = False

if RUN_LOCAL_WHISPER:
    try:
        local_stt = LocalWhisperProvider(model_name="base")
        timed_local = measure_call(local_stt.transcribe, sample_audio)
        local_result = timed_local.value
        local_result.latency_ms = timed_local.elapsed_ms
        print(local_result)
        print(f"Latency: {local_result.latency_ms:.0f} ms")
    except Exception as exc:
        print("Local Whisper skipped:", exc)
else:
    print("Local Whisper is disabled by default for classroom reliability.")


100%|███████████████████████████████████████| 139M/139M [00:42<00:00, 3.43MiB/s]


Local Whisper skipped: [WinError 2] The system cannot find the file specified


C:\Users\DELL\AppData\Roaming\Python\Python314\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


## 4. Generate a speech-safe response

Voice responses should be shorter than chat responses. The answer should avoid markdown, tables, URLs, and long clauses because users hear the response sequentially.


In [7]:
agent_for_text = VoiceAgent(stt_provider=MockSTTProvider(), tts_provider=MockTTSProvider())

raw_question = openai_transcript.text
response_text = agent_for_text.generate_response(raw_question, use_mock_llm=False)

print("Question:", raw_question)
print("Speech-safe response:", response_text)


Question: Can you explain in simple words how text-to-speech works for an AI assistant?
Speech-safe response: Text-to-speech converts written text into spoken words. It uses algorithms to analyze the text and generate human-like speech sounds.


## 5. TTS baseline: mock provider

The mock TTS provider only proves the pipeline writes an audio file. It is not a quality benchmark.


In [9]:
response_text

'Text-to-speech converts written text into spoken words. It uses algorithms to analyze the text and generate human-like speech sounds.'

In [ ]:
mock_tts = MockTTSProvider()
mock_tts_result = mock_tts.synthesize(response_text, OUTPUT_DIR / "mock_response.wav")
print(mock_tts_result)
display(Audio(filename=str(mock_tts_result.output_path)))


TTSResult(output_path=WindowsPath('outputs/mock_response.wav'), provider='mock-tts', model='tone-generator', voice='beep', latency_ms=None)


## 6. OpenAI TTS

OpenAI TTS is the balanced cloud path for this class: simple API, good quality, and easy integration with the rest of the OpenAI-based agent stack.


In [10]:
try:
    openai_tts = OpenAITTSProvider()
    timed_openai_tts = measure_call(openai_tts.synthesize, response_text, OUTPUT_DIR / "openai_response.mp3")
    openai_tts_result = timed_openai_tts.value
    openai_tts_result.latency_ms = timed_openai_tts.elapsed_ms

    print(openai_tts_result)
    print(f"Latency: {openai_tts_result.latency_ms:.0f} ms")
    display(Audio(filename=str(openai_tts_result.output_path)))
except Exception as exc:
    print("OpenAI TTS skipped:", exc)


TTSResult(output_path=WindowsPath('outputs/openai_response.mp3'), provider='openai-tts', model='gpt-4o-mini-tts', voice='marin', latency_ms=3425.484999999753)
Latency: 3425 ms


## 7. ElevenLabs TTS

ElevenLabs is useful when expressiveness and voice character matter. Use the same response text so the comparison is fair.


In [11]:
try:
    eleven_tts = ElevenLabsTTSProvider()
    timed_eleven = measure_call(eleven_tts.synthesize, response_text, OUTPUT_DIR / "elevenlabs_response.mp3")
    eleven_result = timed_eleven.value
    eleven_result.latency_ms = timed_eleven.elapsed_ms

    print(eleven_result)
    print(f"Latency: {eleven_result.latency_ms:.0f} ms")
    display(Audio(filename=str(eleven_result.output_path)))
except Exception as exc:
    print("ElevenLabs TTS skipped:", exc)


TTSResult(output_path=WindowsPath('outputs/elevenlabs_response.mp3'), provider='elevenlabs-tts', model='eleven_multilingual_v2', voice='rOskTrl4fsn0CNC61W1f', latency_ms=7842.854399998032)
Latency: 7843 ms


## 8. Optional: Piper local TTS

Piper is the local/free comparison. It needs a downloaded voice model and matching config file.

Install and configure only if the classroom machine is ready:

```bash
pip install -r requirements_piper_optional.txt
```


In [13]:
RUN_PIPER = True

if RUN_PIPER:
    try:
        piper_tts = PiperTTSProvider()
        timed_piper = measure_call(piper_tts.synthesize, response_text, OUTPUT_DIR / "piper_response.wav")
        piper_result = timed_piper.value
        piper_result.latency_ms = timed_piper.elapsed_ms

        print(piper_result)
        print(f"Latency: {piper_result.latency_ms:.0f} ms")
        display(Audio(filename=str(piper_result.output_path)))
    except Exception as exc:
        print("Piper TTS skipped:", exc)
else:
    print("Piper is disabled by default. Enable RUN_PIPER after installing Piper and setting model paths.")


Piper TTS skipped: # channels not specified


## 9. Integrated voice-agent turn

Now put the pieces together. The same `VoiceAgent` can use mock providers, OpenAI providers, or a mixed stack.


In [16]:
voice_agent = VoiceAgent(
    stt_provider=MockSTTProvider(),
    tts_provider=OpenAITTSProvider(),
)

turn = voice_agent.run_turn(
    input_audio_path=sample_audio,
    output_audio_path=OUTPUT_DIR / "integrated_mock_response.wav",
    use_mock_llm=True,
)

print("Transcript:", turn.transcript.text)
print("Response:", turn.response_text)
print("Output audio:", turn.audio.output_path)
print(f"Total latency: {turn.total_latency_ms:.0f} ms")
display(Audio(filename=str(turn.audio.output_path)))


Transcript: Can you explain in simple words how text to speech works for an AI assistant?
Response: Text to speech turns written words into audio. For a voice agent, it is the final step that makes the assistant feel present.
Output audio: outputs\integrated_mock_response.wav
Total latency: 4044 ms


## 10. Switch the stack

The important production lesson: changing providers should be a configuration change, not a rewrite.

Try these combinations after setting keys:

```python
VoiceAgent(OpenAITranscriptionProvider(), OpenAITTSProvider())
VoiceAgent(OpenAITranscriptionProvider(), ElevenLabsTTSProvider())
VoiceAgent(MockSTTProvider(), PiperTTSProvider())
```


In [ ]:
# Uncomment after configuring keys.
# real_agent = VoiceAgent(
#     stt_provider=OpenAITranscriptionProvider(),
#     tts_provider=OpenAITTSProvider(),
# )
# real_turn = real_agent.run_turn(
#     input_audio_path=sample_audio,
#     output_audio_path=OUTPUT_DIR / "integrated_openai_response.mp3",
# )
# print(real_turn)
# display(Audio(filename=str(real_turn.audio.output_path)))

print("Provider-switching cell ready. Configure .env before enabling real providers.")


## Exercises

### Challenge 1 — Add a voice style switch
Add a parameter called `voice_style` that changes the LLM system prompt before TTS. Try: calm tutor, energetic product demo, formal support agent.

Hint: keep the response short and speech-safe.

### Challenge 2 — Log latency
Save STT latency, LLM latency, TTS latency, and total latency to a CSV file after every turn.

Hint: extend `VoiceTurn` or write a small `log_turn()` helper.

### Challenge 3 — Prepare for avatar integration
Modify the output path naming so every generated response gets a timestamped audio file.

Hint: avatars need predictable file paths or streams.


## Closing

You now have a reusable speech layer for agents.

The next step is to connect this audio output to an avatar persona: the avatar needs clean audio, short response chunks, consistent voice/persona, and predictable latency.
